In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS taxi_catalog.gold;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

df_silver = spark.table("taxi_catalog.silver.yellow_taxi_clean")
display(df_silver.limit(5))

In [0]:
monthly_metrics = (
    df_silver
    .groupBy("pickup_year", "pickup_month")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("passenger_count").alias("avg_passengers")
    )
    .orderBy("pickup_year", "pickup_month")
)

monthly_metrics.write.format("delta").mode("overwrite").saveAsTable("taxi_catalog.gold.monthly_taxi_metrics")
print("Monthly metrics table created successfully!")
display(monthly_metrics)

In [0]:
payment_metrics = (
    df_silver
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("revenue")
    )
    .orderBy(F.desc("revenue"))
)

payment_metrics.write.format("delta").mode("overwrite").saveAsTable("taxi_catalog.gold.payment_metrics")
print("Payment metrics table created successfully!")
display(payment_metrics)

In [0]:
top_pickup_locations = (
    df_silver
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("revenue")
    )
    .orderBy(F.desc("trip_count"))
    .limit(10)
)

top_pickup_locations.write.format("delta").mode("overwrite").saveAsTable("taxi_catalog.gold.top_pickup_locations")
print("Top pickup locations table created successfully!")
display(top_pickup_locations)

In [0]:
%sql
SELECT 
    pickup_year,
    pickup_month,
    total_trips,
    ROUND(total_revenue, 2) AS total_revenue,
    ROUND(avg_trip_distance, 2) AS avg_distance
FROM taxi_catalog.gold.monthly_taxi_metrics
ORDER BY pickup_year, pickup_month;